# Step 2 - Data Cleaning

Based on what I found in notebook 01:
- 9 columns are 100% null → drop
- Lot columns mostly empty → drop
- Valeur fonciere has French comma format and outliers up to 14 billion
- ~40% of rows have no Type local (pure land transactions)
- Date column needs parsing

Going to clean step by step and check row count after each major drop.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data/'

In [ ]:
# reload raw data (same as notebook 01)
years = [2021, 2022, 2023, 2024, 2025]
dfs = []
for year in years:
    df = pd.read_csv(DATA_DIR + f'ValeursFoncieres-{year}.txt', sep='|', low_memory=False)
    df['annee'] = year
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)
print(f'Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns')

## 1. Drop completely useless columns

In [ ]:
# columns that are 100% null - no point keeping these
cols_100_null = [
    'Identifiant de document', 'Reference document',
    '1 Articles CGI', '2 Articles CGI', '3 Articles CGI', '4 Articles CGI', '5 Articles CGI',
    'Identifiant local'
]

# verify before dropping
print('Null % for these columns:')
print((df[cols_100_null].isnull().mean() * 100).round(1))

df.drop(columns=cols_100_null, inplace=True)
print(f'\nAfter drop: {df.shape[1]} columns')

In [ ]:
# lot columns - mostly empty and not really what we need for analysis
# 'Nombre de lots' is 0% null so keeping that one
# 'Surface Carrez du 1er lot' is 91% null but still might matter for apartments
# actually let me just drop all the lot detail columns, they're too sparse

lot_cols_to_drop = [
    '1er lot', 'Surface Carrez du 1er lot',
    '2eme lot', 'Surface Carrez du 2eme lot',
    '3eme lot', 'Surface Carrez du 3eme lot',
    '4eme lot', 'Surface Carrez du 4eme lot',
    '5eme lot', 'Surface Carrez du 5eme lot',
    'No Volume'  # 99.8% null
]

df.drop(columns=lot_cols_to_drop, inplace=True)
print(f'After lot drop: {df.shape[1]} columns')

In [ ]:
# dropping these - too sparse to be useful:
# B/T/Q: 95.5% null - stands for Bâtiment/Tranche/Queue, used for large multi-unit complexes.
#   Almost always null because most transactions are single-unit sales.
# Prefixe de section: 95.3% null - cadastral section prefix, only applies to certain municipalities
# Nature culture speciale: 95.7% null - specifies crop type on agricultural land, irrelevant for most sales
# Code voie: redundant with Voie (same street name, different encoding)

sparse_cols = ['B/T/Q', 'Prefixe de section', 'Nature culture speciale', 'Code voie']
df.drop(columns=sparse_cols, inplace=True)
print(f'After sparse drop: {df.shape[1]} columns')
print(df.columns.tolist())

## 2. Fix data types

In [ ]:
# Valeur fonciere - French decimal format (comma instead of dot)
df['valeur_fonciere'] = (
    df['Valeur fonciere']
    .str.replace(',', '.', regex=False)
    .astype(float)
)
df.drop(columns=['Valeur fonciere'], inplace=True)

print('valeur_fonciere sample:')
print(df['valeur_fonciere'].describe())

In [ ]:
# Date mutation - parse as datetime
df['date_mutation'] = pd.to_datetime(df['Date mutation'], format='%d/%m/%Y', errors='coerce')
df.drop(columns=['Date mutation'], inplace=True)

# how many failed to parse?
bad_dates = df['date_mutation'].isnull().sum()
print(f'Failed date parses: {bad_dates}')
print(df['date_mutation'].dt.year.value_counts().sort_index())

In [ ]:
# rename columns to snake_case - easier to type
rename_map = {
    'No disposition': 'no_disposition',
    'Nature mutation': 'nature_mutation',
    'No voie': 'no_voie',
    'Type de voie': 'type_voie',
    'Voie': 'voie',
    'Code postal': 'code_postal',
    'Commune': 'commune',
    'Code departement': 'code_departement',
    'Code commune': 'code_commune',
    'Section': 'section',
    'No plan': 'no_plan',
    'Nombre de lots': 'nombre_lots',
    'Code type local': 'code_type_local',
    'Type local': 'type_local',
    'Surface reelle bati': 'surface_bati',
    'Nombre pieces principales': 'nb_pieces',
    'Nature culture': 'nature_culture',
    'Surface terrain': 'surface_terrain',
}

df.rename(columns=rename_map, inplace=True)
print(df.columns.tolist())

## 3. Handle missing values in key columns

In [ ]:
# check missing after column drops
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({'count': missing, 'pct': missing_pct})[missing > 0].sort_values('pct', ascending=False)

In [ ]:
# rows with no valeur_fonciere are useless - it's the main value
before = len(df)
df = df[df['valeur_fonciere'].notna()]
print(f'Dropped {before - len(df):,} rows with null valeur_fonciere')
print(f'Remaining: {len(df):,}')

In [ ]:
# type_local is null for ~40% - these are pure terrain/land transactions
# NOT going to drop these, they're valid just different category
# fill with 'Terrain' as a label so we can filter later
df['type_local'] = df['type_local'].fillna('Terrain')
df['type_local'].value_counts()

In [ ]:
# code_type_local - same thing, fill matching
df['code_type_local'] = df['code_type_local'].fillna(0).astype(int)

# surface_bati and nb_pieces - 0 makes sense for Terrain rows, fill with 0
df['surface_bati'] = df['surface_bati'].fillna(0)
df['nb_pieces'] = df['nb_pieces'].fillna(0)

# surface_terrain - same logic
df['surface_terrain'] = df['surface_terrain'].fillna(0)

print('Missing after fills:')
df.isnull().sum()[df.isnull().sum() > 0]

In [ ]:
# code_postal has 0.7% missing - not going to fill, just leave as NaN
# commune/voie also 0.7% - same
# nature_culture 31.8% - these are rows without land use type (buildings), leave as NaN
print('Remaining nulls - acceptable:')
df.isnull().sum()[df.isnull().sum() > 0]

## 4. Handle outliers

In [ ]:
# valeur_fonciere outliers
# max is 14 billion - clearly commercial/industrial, not residential
# median is ~166k which makes sense for France

print('Valeur fonciere distribution:')
print(df['valeur_fonciere'].describe())

print(f'\nHow many > 1M: {(df["valeur_fonciere"] > 1_000_000).sum():,}')
print(f'How many > 10M: {(df["valeur_fonciere"] > 10_000_000).sum():,}')
print(f'How many < 1000: {(df["valeur_fonciere"] < 1000).sum():,}')

In [ ]:
# look at very cheap transactions - what are these?
df[df['valeur_fonciere'] < 100][['commune', 'nature_mutation', 'type_local', 'valeur_fonciere']].head(10)

In [ ]:
# very cheap ones look like symbolic sales or partial rights transfers
# keeping them for now but flagging - TODO: decide later if we want to filter

# for outlier detection use IQR method on valeur_fonciere
Q1 = df['valeur_fonciere'].quantile(0.25)
Q3 = df['valeur_fonciere'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 3 * IQR  # using 3x instead of 1.5x because property prices vary a lot
upper = Q3 + 3 * IQR

print(f'Q1: {Q1:,.0f}  Q3: {Q3:,.0f}  IQR: {IQR:,.0f}')
print(f'Lower bound: {lower:,.0f}')
print(f'Upper bound: {upper:,.0f}')
print(f'Outliers above: {(df["valeur_fonciere"] > upper).sum():,}')
print(f'Outliers below: {(df["valeur_fonciere"] < lower).sum():,}')

In [ ]:
# before committing to 3x IQR, check what 1.5x and 3x actually flag
# want to make sure 3x isn't too permissive and 1.5x isn't too aggressive

thresholds = [1.5, 2.0, 3.0]
results = []

for mult in thresholds:
    lo = Q1 - mult * IQR
    hi = Q3 + mult * IQR
    n_flagged = ((df['valeur_fonciere'] > hi) | (df['valeur_fonciere'] < 1)).sum()
    pct = n_flagged / len(df) * 100
    
    # what type_local are being flagged?
    flagged = df[(df['valeur_fonciere'] > hi) | (df['valeur_fonciere'] < 1)]
    top_type = flagged['type_local'].value_counts().index[0]
    results.append({'multiplier': mult, 'upper_bound': round(hi), 'flagged': n_flagged, 'pct': round(pct,2), 'mostly': top_type})

import pandas as pd
comparison = pd.DataFrame(results)
print(comparison.to_string(index=False))
print()
print("Going with 3x: 1.5x removes too many rows that are probably legitimate high-value commercial sales.")
print("The 3x upper bound is still ~900k which captures the bulk of residential transactions.")

In [ ]:
# flag outliers instead of dropping - easier to filter later
df['is_outlier_valeur'] = (df['valeur_fonciere'] > upper) | (df['valeur_fonciere'] < 1)
print(f'Flagged as outlier: {df["is_outlier_valeur"].sum():,} ({df["is_outlier_valeur"].mean()*100:.1f}%)')

In [ ]:
# surface_bati outliers - let's see the distribution
print(df[df['surface_bati'] > 0]['surface_bati'].describe())
print(f'\nSurface > 10000m²: {(df["surface_bati"] > 10000).sum():,}')

In [ ]:
# nb_pieces - sanity check
print(df['nb_pieces'].value_counts().sort_index().head(20))
print(f'\nnb_pieces > 20: {(df["nb_pieces"] > 20).sum()}')

In [ ]:
# check for duplicate transactions
# same property (no_plan + commune + section) sold on the same date = likely duplicate row
# this happens because one sale can generate multiple rows (one per lot or per type_local)

dup_key = ['date_mutation', 'no_plan', 'commune', 'section', 'valeur_fonciere']
n_exact_dups = df.duplicated(subset=dup_key).sum()
print(f'Exact duplicates (same date/plan/commune/section/valeur): {n_exact_dups:,}')
print(f'({n_exact_dups/len(df)*100:.2f}% of rows)')

# note: many of these are intentional - one sale = multiple rows (e.g. house + garage sold together)
# the valeur_fonciere is repeated on each row of the same transaction
# this is documented in the DVF data specification
# we keep them because each row represents a distinct property unit
# but it's worth flagging for anyone doing price-level analysis

# how many unique transactions (unique date+plan+commune+valeur)?
n_unique_transactions = df.drop_duplicates(subset=dup_key).shape[0]
print(f'\nUnique transactions: {n_unique_transactions:,}')
print(f'Average rows per transaction: {len(df)/n_unique_transactions:.2f}')

## 5. Final check

In [ ]:
print(f'Final shape: {df.shape}')
print(f'\nColumns:')
print(df.dtypes)
print(f'\nMissing:')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# quick distribution plot to visually confirm cleaning worked
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# only non-outlier ventes (regular sales)
mask = (df['nature_mutation'] == 'Vente') & (~df['is_outlier_valeur'])
sample = df[mask]['valeur_fonciere'].sample(50000, random_state=42)

axes[0].hist(sample, bins=100, color='steelblue', edgecolor='none')
axes[0].set_title('Valeur foncière - Ventes (sans outliers)')
axes[0].set_xlabel('Valeur (€)')

axes[1].hist(np.log1p(sample), bins=100, color='coral', edgecolor='none')
axes[1].set_title('Log(Valeur foncière) - looks more normal')
axes[1].set_xlabel('Log(Valeur + 1)')

plt.tight_layout()
plt.savefig('../outputs/plots/valeur_distribution.png', dpi=100)
plt.show()
print('plot saved')

In [ ]:
# save cleaned dataframe
# TODO: try parquet - csv is slow to write for 20M rows
df.to_parquet('../data/cleaned.parquet', index=False)
print(f'Saved cleaned data: {len(df):,} rows')